# BÀI 5. TÔ MÀU ĐỒ THỊ (TO_MAU_TOIUU.*)

## Mô tả bài toán:
Cho n đỉnh (thành phố) và các cạnh biểu diễn mối liên kết giữa chúng. Hãy tô màu các đỉnh sao cho:
- Không có hai đỉnh kề nhau được tô cùng một màu
- Sử dụng **số màu ít nhất có thể** (Sắc số bằng cách thuật toán tham lam hoặc tối ưu)

## Dữ liệu:
**Nhập từ bàn phím:**
- Dòng đầu tiên chứa số đỉnh **N** (N ≤ 1000) và số cạnh **M**
- **M dòng tiếp theo**, mỗi dòng chứa 2 số **u, v** là hai đỉnh u, v kề nhau

## Dữ liệu ra:
**Xuất ra màn hình (cho mỗi phương pháp):**
- Dòng đầu tiên chứa số màu **k** cần tô
- **k dòng tiếp theo**, liệt kê các đỉnh được tô cùng một màu

## Ví dụ:

**Dữ liệu:**
```
5 4
1 2
1 3
2 4
3 4
```

**Kết quả:**
```
THUẬT TOÁN THAM LAM + SẮP XẾP
Số màu cần dùng: 2
Màu 1: 1 4
Màu 2: 2 3 5

THUẬT TOÁN TÔ MÀU TỐI ƯU (QUAY LUI)
Số màu cần dùng: 2
Màu 1: 1 4
Màu 2: 2 3 5
```

## PHƯƠNG PHÁP 1: THUẬT TOÁN THAM LAM + SẮP XẾP (Welsh-Powell)

### Ý tưởng:
1. Tính bậc (degree) của từng đỉnh (số cạnh kề)
2. Sắp xếp các đỉnh theo bậc **giảm dần**
3. Xét từng đỉnh theo thứ tự sắp xếp, tô màu **nhỏ nhất** chưa bị sử dụng bởi các đỉnh kề

### Ưu điểm:
- Độ phức tạp: **O(N + M)** (nhanh)
- Sắp xếp theo bậc giúp tối ưu hơn so với tham lam thuần

### Nhược điểm:
- Không **đảm bảo** là tối ưu (sắc số lý thuyết)
- Kết quả phụ thuộc vào độ tốt của heuristic sắp xếp

In [1]:
def tham_lam_sap_xep(n, adj):
    """
    Thuật toán Welsh-Powell (Tham lam + Sắp xếp).
    
    Args:
        n: Số đỉnh
        adj: Danh sách kề (dictionary)
    
    Returns:
        colors: Dictionary {đỉnh: màu}
    """
    # Tính bậc của từng đỉnh
    degrees = [(len(adj[u]), u) for u in range(1, n + 1)]
    
    # Sắp xếp theo bậc giảm dần
    degrees.sort(reverse=True, key=lambda x: x[0])
    sorted_vertices = [u for deg, u in degrees]
    
    colors = {}
    
    for u in sorted_vertices:
        # Lấy danh sách màu đã bị các đỉnh kề "chiếm dụng"
        used_colors = {colors[v] for v in adj[u] if v in colors}
        
        # Tìm màu nhỏ nhất chưa bị chiếm (bắt đầu từ 1)
        c = 1
        while c in used_colors:
            c += 1
        
        colors[u] = c
    
    return colors

### Test ví dụ với Thuật toán Tham lam + Sắp xếp:

In [2]:
# Ví dụ từ đề bài
print("=== TEST THUẬT TOÁN THAM LAM + SẮP XẾP ===")
print("\nBàn phím (Input):")
print("5 4")
print("1 2")
print("1 3")
print("2 4")
print("3 4")

# Khởi tạo đữ liệu
n = 5
m = 4
edges = [(1, 2), (1, 3), (2, 4), (3, 4)]

# Xây dựng danh sách kề
adj = {i: [] for i in range(1, n + 1)}
for u, v in edges:
    adj[u].append(v)
    adj[v].append(u)

# Chạy thuật toán tham lam
kq_tham_lam = tham_lam_sap_xep(n, adj)

# Gom nhóm theo màu
color_groups = {}
for u, c in kq_tham_lam.items():
    if c not in color_groups:
        color_groups[c] = []
    color_groups[c].append(u)

# In ra kết quả
print("\nMàn hình (Output):")
print(f"Số màu cần dùng: {max(kq_tham_lam.values())}")
for c in sorted(color_groups.keys()):
    print(f"Màu {c}: " + " ".join(map(str, sorted(color_groups[c]))))

=== TEST THUẬT TOÁN THAM LAM + SẮP XẾP ===

Bàn phím (Input):
5 4
1 2
1 3
2 4
3 4

Màn hình (Output):
Số màu cần dùng: 2
Màu 1: 1 4 5
Màu 2: 2 3


## PHƯƠNG PHÁP 2: THUẬT TOÁN TÔ MÀU TỐI ƯU (Quay lui / Backtracking)

### Ý tưởng:
1. Thử giới hạn số màu **m** tăng dần từ **1** đến **N**
2. Dùng **Quay lui (Backtracking)** để kiểm tra xem có thể tô m màu được không
3. Giá trị **m đầu tiên** mà tô thành công là **sắc số tối ưu**

### Ưu điểm:
- **Đảm bảo tối ưu** (sắc số chính xác)
- Tìm được số màu nhỏ nhất lý thuyết

### Nhược điểm:
- Độ phức tạp: **O(N^N)** (chậm cho đồ thị lớn)
- Chỉ thích hợp với N ≤ 20-30

In [3]:
def to_mau_toi_uu(n, adj):
    """
    Thuật toán tô màu tối ưu (Quay lui / Backtracking).
    
    Args:
        n: Số đỉnh
        adj: Danh sách kề (dictionary)
    
    Returns:
        color: Dictionary {đỉnh: màu} khi tìm được hoặc {} nếu không tìm
    """
    
    def is_safe(u, color, c):
        """Kiểm tra liệu có thể tô đỉnh u bằng màu c không"""
        for v in adj[u]:
            if color.get(v) == c:
                return False
        return True

    def Try(u, m, color):
        """Quay lui để tô đỉnh u với tối đa m màu"""
        if u > n:  # Đã tô xong tất cả n đỉnh
            return True
        
        for c in range(1, m + 1):
            if is_safe(u, color, c):
                color[u] = c
                if Try(u + 1, m, color):
                    return True
                color[u] = 0  # Backtrack (hoàn tác quyết định)
        
        return False

    # Thử lần lượt số màu từ 1 đến n
    for m in range(1, n + 1):
        color = {i: 0 for i in range(1, n + 1)}
        if Try(1, m, color):
            return color
    
    return {}

### Test ví dụ với Thuật toán Tô màu tối ưu:

In [4]:
# Sử dụng cùng ví dụ
print("\n=== TEST THUẬT TOÁN TÔ MÀU TỐI ƯU (QUAY LUI) ===")

# Chạy thuật toán tối ưu
kq_toi_uu = to_mau_toi_uu(n, adj)

# Gom nhóm theo màu
color_groups_optimal = {}
for u, c in kq_toi_uu.items():
    if c not in color_groups_optimal:
        color_groups_optimal[c] = []
    color_groups_optimal[c].append(u)

# In ra kết quả
if kq_toi_uu:
    print(f"Số màu cần dùng: {max(kq_toi_uu.values())}")
    for c in sorted(color_groups_optimal.keys()):
        print(f"Màu {c}: " + " ".join(map(str, sorted(color_groups_optimal[c]))))
else:
    print("Không tìm được lời giải")


=== TEST THUẬT TOÁN TÔ MÀU TỐI ƯU (QUAY LUI) ===
Số màu cần dùng: 2
Màu 1: 1 4 5
Màu 2: 2 3


## SO SÁNH HAI PHƯƠNG PHÁP

| Tiêu chí | Tham lam + Sắp xếp | Tối ưu (Backtracking) |
|----------|--------------------|-----------------------|
| **Độ chính xác** | Heuristic (không đảm bảo) | Luôn tối ưu ✓ |
| **Tốc độ** | O(N² + M) | O(N^N) |
| **Kích thước đồ thị** | Ngàn đỉnh | ≤ 20-30 đỉnh |
| **Ứng dụng** | Bài toán thực tế lớn | Bài toán nhỏ, cần tối ưu |

### Khi nào dùng cái nào?
- **Tham lam + Sắp xếp**: Khi đồ thị lớn (N > 30), cần giải nhanh, không yêu cầu tối ưu
- **Tối ưu (Backtracking)**: Khi đồ thị nhỏ (N ≤ 30), **cần sắc số chính xác**

### Nhập dữ liệu từ người dùng và chạy cả 2 phương pháp:

In [5]:
def in_ket_qua(ten_phuong_phap, colors):
    """Hàm tiện ích để in kết quả"""
    if not colors:
        print(f"\n--- {ten_phuong_phap} ---")
        print("Không tìm được lời giải")
        return
    
    color_groups = {}
    for u, c in colors.items():
        if c not in color_groups:
            color_groups[c] = []
        color_groups[c].append(u)
    
    k = max(color_groups.keys())
    print(f"\n--- {ten_phuong_phap} ---")
    print(f"Số màu cần dùng: {k}")
    for c in range(1, k + 1):
        if c in color_groups:
            print(f"Màu {c}: " + " ".join(map(str, sorted(color_groups[c]))))

# Nhập dữ liệu từ người dùng
print("Nhập dữ liệu đồ thị:")
n, m = map(int, input("Nhập N (số đỉnh) và M (số cạnh): ").split())

adj = {i: [] for i in range(1, n + 1)}
for _ in range(m):
    u, v = map(int, input("Nhập u, v (cạnh u-v): ").split())
    adj[u].append(v)
    adj[v].append(u)

# Chạy và so sánh 2 phương pháp
print("\n" + "="*50)
kq_tham_lam = tham_lam_sap_xep(n, adj)
in_ket_qua("THUẬT TOÁN THAM LAM + SẮP XẾP", kq_tham_lam)

kq_toi_uu = to_mau_toi_uu(n, adj)
in_ket_qua("THUẬT TOÁN TÔ MÀU TỐI ƯU (QUAY LUI)", kq_toi_uu)
print("="*50)

Nhập dữ liệu đồ thị:


--- THUẬT TOÁN THAM LAM + SẮP XẾP ---
Số màu cần dùng: 2
Màu 1: 1 4
Màu 2: 2 3

--- THUẬT TOÁN TÔ MÀU TỐI ƯU (QUAY LUI) ---
Số màu cần dùng: 2
Màu 1: 1 4
Màu 2: 2 3
